In [1]:
import sys
sys.path.append('../../')

In [9]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [17]:
import numpy as np
import pandas as pd

from collections import Counter

import torch
import torchtext

from src.config.paths import DATA_PATH, PRETRAINED_PATH
from src.core.text.summarizer import BasicEnglishSummarizer

# Loading and Preproc

## GloVe Embeddings

In [18]:
model = BasicEnglishSummarizer()

In [34]:
dict(enumerate(model.vocab.itos))

{0: '<s>',
 1: '</s>',
 2: '<unk>',
 3: '<pad>',
 4: ',',
 5: '.',
 6: 'the',
 7: 'and',
 8: 'to',
 9: 'of',
 10: 'a',
 11: 'in',
 12: '"',
 13: ':',
 14: 'is',
 15: 'for',
 16: 'I',
 17: ')',
 18: '(',
 19: 'that',
 20: '-',
 21: 'on',
 22: 'you',
 23: 'with',
 24: "'s",
 25: 'it',
 26: 'The',
 27: 'are',
 28: 'by',
 29: 'at',
 30: 'be',
 31: 'this',
 32: 'as',
 33: 'from',
 34: 'was',
 35: 'have',
 36: 'or',
 37: '...',
 38: 'your',
 39: 'not',
 40: '!',
 41: '?',
 42: 'will',
 43: 'an',
 44: "n't",
 45: 'can',
 46: 'but',
 47: 'all',
 48: 'my',
 49: 'has',
 50: '|',
 51: 'do',
 52: 'we',
 53: 'they',
 54: 'more',
 55: 'one',
 56: 'about',
 57: 'he',
 58: ';',
 59: "'",
 60: 'out',
 61: '$',
 62: 'their',
 63: 'so',
 64: 'his',
 65: 'up',
 66: 'It',
 67: '&',
 68: 'like',
 69: '/',
 70: '1',
 71: 'which',
 72: 'if',
 73: 'would',
 74: 'our',
 75: '[',
 76: ']',
 77: 'me',
 78: 'who',
 79: 'just',
 80: 'This',
 81: 'time',
 82: 'what',
 83: 'A',
 84: '2',
 85: 'had',
 86: 'when',
 87:

In [ ]:
{v:k for k,v in enumerate(model.vocab.itos)}

{'<s>': 566232,
 '</s>': 437252,
 '<unk>': 2,
 '<pad>': 3,
 ',': 4,
 '.': 5,
 'the': 6,
 'and': 7,
 'to': 8,
 'of': 9,
 'a': 10,
 'in': 11,
 '"': 12,
 ':': 13,
 'is': 14,
 'for': 15,
 'I': 16,
 ')': 17,
 '(': 18,
 'that': 19,
 '-': 20,
 'on': 21,
 'you': 22,
 'with': 23,
 "'s": 24,
 'it': 25,
 'The': 26,
 'are': 27,
 'by': 28,
 'at': 29,
 'be': 30,
 'this': 31,
 'as': 32,
 'from': 33,
 'was': 34,
 'have': 35,
 'or': 36,
 '...': 37,
 'your': 38,
 'not': 39,
 '!': 40,
 '?': 41,
 'will': 42,
 'an': 43,
 "n't": 44,
 'can': 45,
 'but': 46,
 'all': 47,
 'my': 48,
 'has': 49,
 '|': 50,
 'do': 51,
 'we': 52,
 'they': 53,
 'more': 54,
 'one': 55,
 'about': 56,
 'he': 57,
 ';': 58,
 "'": 59,
 'out': 60,
 '$': 61,
 'their': 62,
 'so': 63,
 'his': 64,
 'up': 65,
 'It': 66,
 '&': 67,
 'like': 68,
 '/': 69,
 '1': 70,
 'which': 71,
 'if': 72,
 'would': 73,
 'our': 74,
 '[': 75,
 ']': 76,
 'me': 77,
 'who': 78,
 'just': 79,
 'This': 80,
 'time': 81,
 'what': 82,
 'A': 83,
 '2': 84,
 'had': 85,
 'when'

tensor([ 4.4213e-01, -1.3069e-01,  1.3177e-01, -2.0734e-02,  1.8385e-01,
        -2.4139e-01,  4.2682e-01,  6.6350e-01, -4.2280e-01, -1.3182e+00,
         8.9690e-01,  6.9487e-02,  1.8833e-01,  6.4092e-01,  7.5374e-01,
         4.7922e-01,  1.5555e-01, -1.3322e+00, -4.4204e-01,  3.5286e-01,
        -4.9904e-01,  2.2915e-01,  3.3414e-01, -5.6617e-02, -4.2288e-01,
         3.0555e-01, -4.3267e-01,  2.1671e-01, -1.7978e-01,  2.9344e-03,
        -6.1375e-02, -2.3775e-01,  3.4494e-01,  1.3399e-01, -1.6143e-01,
        -1.4828e-01,  8.3514e-01, -4.7450e-02, -2.7438e-01,  9.4555e-01,
        -3.2422e-02,  3.6721e-01, -3.4479e-02, -2.6252e-01, -3.2278e-01,
        -4.9477e-01,  1.6184e-02, -3.4080e-01, -1.3057e-01,  2.6092e-01,
         3.1645e-01,  1.4784e-01,  8.8557e-01,  1.7651e-01,  6.4041e-01,
        -1.2515e+00,  1.5421e-01,  1.7217e-01, -3.2360e-02, -1.2450e-01,
        -3.1818e-02, -8.6255e-01,  5.4778e-01, -4.3682e-01, -5.8938e-01,
         3.1747e-01, -8.2669e-02, -5.6269e-04, -3.1

## Dataset

In [7]:
train_df = pd.read_parquet(DATA_PATH.__str__() + '/xsum/data/train-00000-of-00001.parquet')
test_df = pd.read_parquet(DATA_PATH.__str__() + '/xsum/data/test-00000-of-00001.parquet')
val_df = pd.read_parquet(DATA_PATH.__str__() + '/xsum/data/validation-00000-of-00001.parquet')

train_df = train_df.drop(['id'], axis=1)
test_df = test_df.drop(['id'], axis=1)
val_df = val_df.drop(['id'], axis=1)

In [ ]:
class SummarizationDataset(torch.utils.data.Dataset):
    def __init__(self, df, vocab):
        tokenizer = torchtext.data.get_tokenizer('basic_english')
        
        self.x = [[k if k in vocab else '<unk>' for k in tokenizer(i)] for i in df['document']]
        self.y = [[k if k in vocab else '<unk>' for k in tokenizer(i)] for i in df['summary']]

    def __getitem__(self, idx: int) -> tuple:
        return self.x[idx], self.y[idx]

    def __len__(self):
        return len(self.df)

In [10]:
train_set = SummarizationDataset(train_df, vocab)
test_set = SummarizationDataset(test_df, vocab)
val_set = SummarizationDataset(val_df, vocab)

KeyboardInterrupt: 

In [ ]:
train_set['word']

NameError: name 'train_set' is not defined